# 01 · 数据与可视化：收益、波动与回撤

**学习目标**：把价格序列变成投资语言：日收益率、累计收益、年化波动、最大回撤。

**数据**：默认使用 `scripts/optimize_portfolio.py` 内置的可复现示例价格（3 个资产、250 个交易日），**无需联网**。想换成真实数据，见文末“换成真实数据”。

> 本仓库仅供学习，所有内容不构成投资建议。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 让 Notebook 能直接复用 scripts/ 里的函数
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
pd.set_option("display.float_format", "{:.4f}".format)

## 1. 载入价格

`load_prices()` 在 `data/price_history.csv` 不存在时生成固定随机种子的示例价格，所以每次运行结果相同。

In [ ]:
from optimize_portfolio import load_prices

prices = load_prices()
print(prices.shape)
prices.head()

## 2. 日收益率与累计收益

- 日收益率 `r_t = P_t / P_{t-1} - 1`
- 累计收益（净值）`NAV_t = ∏(1 + r_i)`

In [ ]:
rets = prices.pct_change().dropna()
nav = (1 + rets).cumprod()

ax = nav.plot(figsize=(9, 4), title="Cumulative return (NAV)")
ax.set_ylabel("NAV")
plt.show()

## 3. 年化波动与最大回撤

- 年化波动 = 日收益标准差 × √252
- 最大回撤 = 净值相对历史最高点的最大跌幅，衡量“最难熬的时候亏多少”。

In [ ]:
def perf_stats(daily_ret: pd.Series, periods: int = 252) -> pd.Series:
    """年化收益、年化波动、夏普（无风险利率按 0）、最大回撤。"""
    daily_ret = daily_ret.dropna()
    nav = (1 + daily_ret).cumprod()
    ann_ret = nav.iloc[-1] ** (periods / len(daily_ret)) - 1
    ann_vol = daily_ret.std() * np.sqrt(periods)
    max_dd = (nav / nav.cummax() - 1).min()
    return pd.Series({"ann_return": ann_ret, "ann_vol": ann_vol,
                      "sharpe": ann_ret / ann_vol if ann_vol else np.nan, "max_drawdown": max_dd})

In [ ]:
stats = rets.apply(perf_stats)
stats

In [ ]:
drawdown = nav / nav.cummax() - 1
ax = drawdown.plot(figsize=(9, 3), title="Drawdown")
ax.set_ylabel("drawdown")
plt.show()

## 4. 相关性

分散化的效果取决于资产之间的相关性：相关性越低，组合波动越能被压低。

In [ ]:
rets.corr()

## 练习

1. 把年化因子 252 改成 52（周频）或 12（月频）时，应该先对收益做什么处理？
2. 计算等权组合（三个资产各 1/3、每日再平衡）的 `perf_stats`，和单个资产比，波动和回撤有什么变化？
3. 找出最大回撤发生的起止日期。

## 换成真实数据

把价格保存为 `data/price_history.csv`（列：`date` + 各资产收盘价），`load_prices()` 会自动读取。例如在国内网络下用 AkShare：

```python
import akshare as ak
df = ak.stock_zh_a_hist(symbol="600519", period="daily", start_date="20240101", adjust="qfq")
```